# PyBiRewireX — Tutorial

This notebook mirrors the workflow of the BiRewire R package vignette (`BiRewire.Rnw`).

**What PyBiRewireX does:**  
Given a binary network (bipartite or undirected), generate randomised versions
that are uniformly drawn from the set of all networks with the same degree sequence.
This is the standard null model for testing whether observed network properties
are driven by degree alone.

**Algorithm:** Switching Algorithm (SA) — at each step, two edges are chosen at
random and their endpoints are swapped if the result is a valid edge rewiring.
After N switching steps the distribution is provably close to uniform.

---

## Contents

1. [Installation](#1-installation)
2. [Create networks](#2-create-networks)
3. [Bipartite rewiring](#3-bipartite-rewiring)
4. [Convergence analysis](#4-convergence-analysis)
5. [Undirected rewiring](#5-undirected-rewiring)
6. [Sparse and graph inputs](#6-sparse-and-graph-inputs)
7. [Sampler — null model collections](#7-sampler)
8. [Markov chain monitoring (t-SNE)](#8-markov-chain-monitoring)

## 1. Installation

```bash
pip install pybirewirex            # core
pip install "pybirewirex[graph]"   # + igraph / networkx support
pip install "pybirewirex[vis]"     # + matplotlib / scikit-learn for plots
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

import pybirewirex as pbr
from pybirewirex.similarity import jaccard
from pybirewirex._bounds import bound_bipartite

print(f"C backend available: {pbr._C_AVAILABLE}")

## 2. Create networks

Matching the R vignette example:
```r
ggg   <- bipartite.random.game(n1=100, n2=40, p=0.2)
g.und <- erdos.renyi.game(directed=F, loops=F, n=200, p.or.m=0.05)
```

In [ ]:
SEED = 42
rng  = np.random.default_rng(SEED)

# 100×40 bipartite incidence matrix at 20% density
bp   = (rng.random((100, 40)) < 0.20).astype(np.int16)
e_bp = int(bp.sum())
print(f"Bipartite:  {bp.shape}, {e_bp} edges ({e_bp/bp.size:.1%})")

# 200-node undirected Erdős–Rényi at 5% density
_u  = (rng.random((200, 200)) < 0.05).astype(np.int16)
_u  = np.triu(_u, k=1)
und = (_u + _u.T).clip(0, 1).astype(np.int16)
e_und = int(und.sum()) // 2
print(f"Undirected: {und.shape[0]} nodes, {e_und} edges ({e_und/(und.shape[0]*(und.shape[0]-1)//2):.1%})")

## 3. Bipartite rewiring

```r
# R vignette
m2 <- birewire.rewire.bipartite(BRCA_binary_matrix, verbose=FALSE)
```

In [ ]:
bp_rw = pbr.rewire_bipartite(bp, verbose=False, seed=SEED)

print("Row sums preserved:", np.array_equal(bp.sum(1), bp_rw.sum(1)))
print("Col sums preserved:", np.array_equal(bp.sum(0), bp_rw.sum(0)))
print(f"Jaccard(original, rewired) = {jaccard(bp, bp_rw):.4f}")

The Jaccard similarity after `max_iter='n'` (default) switching steps is close
to the stationary value $e / (2|E| - e)$ where $e = |E|$.

In [ ]:
stationary = e_bp / (2 * e_bp - e_bp)   # simplifies to 1 — but let's do it properly
stationary = e_bp / (bp.size * 2 - e_bp)  # this is wrong; correct:
# Stationary Jaccard ≈ e / (2*|E| - e) where |E| = number of edges in the union of two i.i.d. samples
# For large networks: E[Jaccard] ≈ d / (2 - d) where d = density
d = e_bp / bp.size
print(f"Expected stationary Jaccard ≈ {d/(2-d):.4f}")

## 4. Convergence analysis

```r
# R vignette
N   <- birewire.analysis.bipartite(get.incidence(ggg), max.iter=2, step=1)$N
res <- birewire.analysis.bipartite(get.incidence(ggg), max.iter=10*N, n.networks=10)
```

In [ ]:
N_bp   = bound_bipartite(e_bp, bp.size, accuracy=0.00005, exact=False)
print(f"Analytical bound N = {N_bp}")

step   = max(1, 10 * N_bp // 30)
result = pbr.analysis_bipartite(
    bp, n_networks=10, max_iter=10*N_bp, step=step, verbose=False, seed=SEED,
)
print(f"Scores: {result.scores.shape}  (n_networks × n_steps)")

In [ ]:
xs   = np.arange(result.scores.shape[1]) * result.step
mean = result.scores.mean(0)
ci   = 1.96 * result.scores.std(0, ddof=1) / np.sqrt(result.scores.shape[0])

fig, ax = plt.subplots(figsize=(8, 4))
for row in result.scores:
    ax.plot(xs, row, color="#1f77b4", alpha=0.2, lw=0.8)
ax.plot(xs, mean, color="#1f77b4", lw=2, label="mean")
ax.fill_between(xs, mean-ci, mean+ci, alpha=0.25, color="#1f77b4", label="95 % CI")
ax.axvline(result.N, color="k", ls="--", lw=1.2, label=f"N = {result.N}")
ax.set(xlabel="Switching steps", ylabel="Jaccard similarity", ylim=(0, 1.05))
ax.legend(); ax.grid(ls=":", alpha=0.4)
fig.suptitle("Bipartite convergence — analogue of R vignette analysis.pdf")
plt.tight_layout()
plt.show()

## 5. Undirected rewiring

```r
# R vignette
m2.und <- birewire.rewire.undirected(m.und, verbose=FALSE)
```

In [ ]:
und_rw = pbr.rewire_undirected(und, verbose=False, seed=SEED)

print("Degrees preserved:", np.array_equal(und.sum(0), und_rw.sum(0)))
print("Symmetric:        ", np.array_equal(und_rw, und_rw.T))
print(f"Jaccard(original, rewired) = {jaccard(und, und_rw):.4f}")

# Convergence analysis
t_und  = und.shape[0] * (und.shape[0] - 1) // 2
N_und  = bound_bipartite(e_und, t_und, accuracy=0.00005, exact=False)
step_u = max(1, 10 * N_und // 30)
res_u  = pbr.analysis_undirected(
    und, n_networks=5, max_iter=10*N_und, step=step_u, verbose=False, seed=SEED,
)
print(f"\nUndirected N = {N_und},  scores: {res_u.scores.shape}")

## 6. Sparse and graph inputs

All rewiring functions accept `scipy.sparse` matrices and graph objects,
returning the same type as the input.

In [ ]:
import scipy.sparse as sp

bp_sparse = sp.csr_matrix(bp)
rw_sparse = pbr.rewire_bipartite(bp_sparse, verbose=False, seed=SEED)
print(f"Input type:  {type(bp_sparse).__name__}")
print(f"Output type: {type(rw_sparse).__name__}")
print(f"Row sums preserved: {np.array_equal(np.asarray(bp_sparse.sum(1)).ravel(), np.asarray(rw_sparse.sum(1)).ravel())}")

In [ ]:
try:
    import igraph as ig
    types = [False] * 100 + [True] * 40
    edges = [(i, 100 + j) for i in range(100) for j in range(40)
             if np.random.default_rng(1).random() < 0.2]
    g    = ig.Graph.Bipartite(types, edges)
    g_rw = pbr.rewire_bipartite(g, verbose=False, seed=SEED)
    print(f"igraph input → {type(g_rw).__name__}, {g_rw.ecount()} edges")
except ImportError:
    print("igraph not installed — pip install 'pybirewirex[graph]'")

## 7. Sampler

Generate K null-model networks, each N switching steps beyond the previous one.

```r
# R vignette
birewire.sampler.bipartite(ggg, K=10000, path="TESTBIREWIREBIPARTITE")
```

In [ ]:
K       = 10
samples = []
current = bp.copy()
for k in range(K):
    current = pbr.rewire_bipartite(current, verbose=False, seed=SEED + k + 1)
    samples.append(current.copy())

# Verify all samples preserve the degree sequence
all_ok = all(
    np.array_equal(s.sum(1), bp.sum(1)) and np.array_equal(s.sum(0), bp.sum(0))
    for s in samples
)
print(f"{K} samples generated; all degree-preserving: {all_ok}")

# Pairwise Jaccard among samples
pw = [jaccard(samples[i], samples[j]) for i in range(K) for j in range(i+1, K)]
print(f"Pairwise Jaccard: mean={np.mean(pw):.3f}  min={np.min(pw):.3f}  max={np.max(pw):.3f}")

## 8. Markov chain monitoring

Visualise the Markov chain by sampling networks at different intervals and
projecting pairwise Jaccard distances into 2-D via t-SNE.

```r
# R vignette
tsne = birewire.visual.monitoring.bipartite(
           ggg, display=T, n.networks=75,
           sequence=c(1,10,200,500,"n",10000), ncol=3, perplexity=10)
```

**R vs Python note:** R calls `Rtsne(m, perplexity=p, check_duplicates=FALSE)` without
`is_distance=TRUE`, so Rtsne treats each row as a feature vector and runs:
normalize → PCA(50) → t-SNE(euclidean). We replicate this pipeline exactly.

In [ ]:
def collect_chain(start, interval, n_samples, base_seed):
    nets    = [start.copy()]
    current = start.copy()
    for i in range(n_samples - 1):
        current = pbr.rewire_bipartite(
            current, max_iter=interval, verbose=False, seed=base_seed + i)
        nets.append(current.copy())
    return nets

def pairwise_dist(nets):
    n = len(nets)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            D[i, j] = D[j, i] = 1.0 - jaccard(nets[i], nets[j])
    return D

def embed_tsne(D, perplexity=10, seed=42):
    D_n   = D / (np.abs(D).max() + 1e-12)
    n_pca = min(50, D_n.shape[1] - 1)
    D_pca = PCA(n_components=n_pca, random_state=seed).fit_transform(D_n)
    return TSNE(n_components=2, metric="euclidean", perplexity=perplexity,
                random_state=seed, init="random", max_iter=1000).fit_transform(D_pca)

In [ ]:
# Smaller n_networks for notebook speed; use scripts/demo.py for full n=75
N_MON    = 30
SEQUENCE = [1, 10, 200, 500, N_bp, 10000]
LABELS   = ["1", "10", "200", "500", "N", "10 000"]

embeddings = []
for k, iv in enumerate(SEQUENCE):
    print(f"k={LABELS[k]:<7}", end=" ", flush=True)
    nets = collect_chain(bp, iv, N_MON, base_seed=SEED + k * 10000)
    D    = pairwise_dist(nets)
    embeddings.append(embed_tsne(D, perplexity=6, seed=SEED))
    print("done")

In [ ]:
cmap = plt.cm.coolwarm
norm = Normalize(vmin=0, vmax=N_MON - 1)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
fig.suptitle(
    "Markov chain monitoring — analogue of R vignette monitoring.pdf",
    fontsize=12,
)

for ax, emb, lbl in zip(axes.ravel(), embeddings, LABELS):
    colors = cmap(norm(np.arange(N_MON)))
    ax.scatter(emb[1:, 0], emb[1:, 1], c=colors[1:], s=12, linewidths=0)
    ax.scatter(emb[0, 0], emb[0, 1], color=colors[0], s=60,
               edgecolors="k", linewidths=0.8)
    ax.text(emb[0, 0], emb[0, 1], " start", fontsize=8)
    ax.set_title(f"k = {lbl}", fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel("A.U."); ax.set_ylabel("A.U.")

# Colorbar
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=axes, fraction=0.02, pad=0.04)
cb.set_label("sampling order")
cb.set_ticks([0, N_MON//2, N_MON-1])
cb.set_ticklabels(["first", "mid", "last"])

plt.tight_layout()
plt.show()

---

## Citation

If you use PyBiRewireX, please cite the original papers:

> Gobbi et al. (2014). *Fast randomization of large genomic datasets while
> preserving alteration counts.* Bioinformatics, 30(17), i617–i623.
> https://doi.org/10.1093/bioinformatics/btu474

> Iorio et al. (2016). *Efficient randomization of biological networks while
> preserving functional characterization of individual nodes.*
> BMC Bioinformatics, 17, 542.
> https://doi.org/10.1186/s12859-016-1402-1